### STT Test for Qwen3-ASR vLLM Server

In [1]:
import json
import os
import re
import shutil
import subprocess
import tempfile
import time
from pathlib import Path

import yt_dlp

# ASR 서버 및 테스트 스크립트의 기본 파라미터 설정 (환경변수 우선)
BASE_URL = os.environ.get("STT_BASE_URL", "http://localhost:8080").rstrip("/")
MODEL = os.environ.get("STT_MODEL", "qwen3-asr")
LANGUAGE = os.environ.get("DEFAULT_LANGUAGE", "ko")
RESPONSE_FORMAT = os.environ.get("STT_RESPONSE_FORMAT", "verbose_json")
TIMESTAMP_GRANULARITIES = [
    x.strip()
    for x in os.environ.get("STT_TIMESTAMP_GRANULARITIES", "segment").split(",")
    if x.strip()
]

NOISE_PATTERNS = [
    re.compile(r"\blanguage\s+[A-Za-z][A-Za-z_-]*\s*<\s*asr[\s_-]*text\s*>", re.IGNORECASE),
    re.compile(r"language\s*(?:Korean\s*asr\s*text|Koreanasrtext)", re.IGNORECASE),
    re.compile(r"Korean\s*asr\s*text", re.IGNORECASE),
    re.compile(r"Koreanasrtext", re.IGNORECASE),
    re.compile(
        r"(?:(?<=^)|(?<=[\s\]\)])|(?<=[\uac00-\ud7af0-9]))language(?=$|[\s\uac00-\ud7af0-9])",
        re.IGNORECASE,
    ),
]


def ensure_command(name: str):
    if not shutil.which(name):
        raise RuntimeError(f"Required command not found in PATH: {name}")


def clean_asr_text(text):
    cleaned = text
    for pattern in NOISE_PATTERNS:
        cleaned = pattern.sub("", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned)
    cleaned = re.sub(r"\s+([,.;:!?])", r"\1", cleaned)
    return cleaned.strip()


def build_clean_transcript(payload):
    segments = payload.get("segments")
    if not segments:
        return clean_asr_text(str(payload.get("text", "")))

    paragraphs = []
    current = []
    previous_end = None

    # 보통 이 함수는 아주 크지는 않지만 payload가 크면 오래 걸릴 수 있다. 
    start_t = time.time()

    for segment in segments:
        text = clean_asr_text(str(segment.get("text", "")))
        if not text:
            continue

        start = segment.get("start")
        if current and previous_end is not None and start is not None and float(start) - previous_end >= 2.5:
            paragraphs.append(" ".join(current))
            current = []

        current.append(text)
        if segment.get("end") is not None:
            previous_end = float(segment["end"])

    if current:
        paragraphs.append(" ".join(current))
    end_t = time.time()
    elapsed = end_t - start_t
    if elapsed > 3.0:  # 3초 이상 걸렸으면 경고 출력
        print(f"[TIME] build_clean_transcript took {elapsed:.2f} seconds")
    return "\n\n".join(paragraphs)


def download_audio(youtube_url, target_dir):
    ydl_opts = {
        "outtmpl": str(target_dir / "audio.%(ext)s"),
        "format": "bestaudio/best",
        "noplaylist": True,
        "quiet": True,
        "continuedl": False,
        "nooverwrites": True,
    }
    print("[INFO] Downloading full audio with yt-dlp...")
    start_t = time.time()
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        audio_path = Path(ydl.prepare_filename(info))
    end_t = time.time()
    elapsed = end_t - start_t
    print(f"[TIME] Audio download took {elapsed:.2f} seconds")

    if not audio_path.is_file():
        raise RuntimeError(f"Downloaded file not found: {audio_path}")
    print(f"[INFO] downloaded_audio={audio_path}")
    return audio_path


def request_transcription(audio_path):
    ensure_command("curl")
    form_args = [
        "-F", f"file=@{audio_path}",
        "-F", f"model={MODEL}",
        "-F", f"language={LANGUAGE}",
        "-F", f"response_format={RESPONSE_FORMAT}",
    ]
    for granularity in TIMESTAMP_GRANULARITIES:
        form_args.extend(["-F", f"timestamp_granularities[]={granularity}"])

    print("[INFO] Requesting STT transcription from server...")
    start_t = time.time()
    proc = subprocess.run(
        [
            "curl",
            "-sS",
            "--fail-with-body",
            "-X", "POST",
            *form_args,
            f"{BASE_URL}/v1/audio/transcriptions",
        ],
        capture_output=True,
        text=True,
        check=False,
        timeout=7200,
    )
    end_t = time.time()
    elapsed = end_t - start_t
    print(f"[TIME] STT transcription request took {elapsed:.2f} seconds")

    if proc.returncode != 0:
        message = proc.stderr.strip() or proc.stdout.strip()
        raise RuntimeError(f"STT request failed: {message}")

    payload = json.loads(proc.stdout or "{}")
    if not isinstance(payload, dict):
        raise RuntimeError(f"Unexpected transcription response type: {type(payload).__name__}")
    return payload

In [2]:
# 사용 예시 (유튜브 링크를 코드 내에서 지정)
youtube_url = "https://www.youtube.com/watch?v=tbndmgWiFJk"  # 원하는 유튜브 링크로 교체 가능
#youtube_url = "https://www.youtube.com/watch?v=OsdI-OzZblo"

with tempfile.TemporaryDirectory(prefix="yt-qwen-asr-jp-") as tmp_dir:
    work_dir = Path(tmp_dir)
    try:
        audio_path = download_audio(youtube_url, work_dir)
        result = request_transcription(audio_path)
        clean_text = build_clean_transcript(result)
        print("\n======= Clean Transcript =======\n")
        print(clean_text)
    except Exception as e:
        print(f"[FAIL] {type(e).__name__}: {e}")

[INFO] Downloading full audio with yt-dlp...


[TIME] Audio download took 3.10 seconds                    
[INFO] downloaded_audio=/tmp/yt-qwen-asr-jp-xjjjpclv/audio.webm
[INFO] Requesting STT transcription from server...
[TIME] STT transcription request took 108.26 seconds

======= Clean Transcript =======

지금 문제는 가장 앞선 세대의 HBM만으로도 이제는 웬만한 수준의 토큰을 처리하기가 어려워지고 있다. 지난번에 나오셔가지고 클로드를 사용을 하는데 한 달에 한 수백만 원을 쓰신다고 했는데 문득 궁금해지더라고요. 클로드를 어느 정도까지 쓸까? 제가 이제 주로 쓰는 것은 이제 클로드 코드 쪽이고요. 10만 줄짜리 코드가 만 줄 이하로 바뀌었어요. 그리고 저는 한 달 정도 돌리는 계산들이 있거든요. 근데 돌리고 나니까 하루에 끝나는 거예요. 정말 뭐 속된 말로 돈 값을 한다 생각합니다. 결국 이 토큰이라는 것이 토큰의 양이 많아지면 많아질수록 메모리도 그만큼 많이 필요하겠다. 그러니까 조금씩 컨센서스가 바뀌고 있는 것이기도 하죠. 들수록 똑똑해지는 지식 뉴스. AI 패러다임이 학습에서 추론 중심으로 이동을 하면서 메모리. 수요가 크게 늘 거다라는 전망은 많았는데 SK 하이닉스도 사상 최대 실적을 기록했다고 합니다. 메모리가 이 정도로 지금 초호황인 건가요? 네, 그렇습니다. 잘 알려져 있는 것처럼 지금은 슈퍼 사이클이라고 볼 수 있는. 뭐 사실은 지난 엔비디아의 GTC라고 해서 엔비디아의 가장 큰 컨퍼런스 행사에서도 젠슨 황이 언급을 한 바 있습니다만 같은 AI 데이터 센터라고 하더라도 이제는 과거에는 누가 더 큰 모델을 만들어서 학습을 시키느냐 이게 훨씬 중요했다면. 최근에 들어와서는 그보다는 추론이 훨씬 더 중요해지고 있다. 그게 열 배나 백 배까지도 더 커질 것이다라고 젠슨 왕이 직접 언급을 한 바

In [3]:
import re
import os

# 디렉토리 생성 (없으면)
output_dir = "/home/kjh/workspace/ExtracTube/notebooks/outputs"
os.makedirs(output_dir, exist_ok=True)

# 유튜브 URL에서 v= 뒷부분(즉, Video ID) 추출
match = re.search(r"v=([A-Za-z0-9_\-]+)", youtube_url)
video_id = match.group(1) if match else "unknown_video"

output_path = os.path.join(output_dir, f"{video_id}.txt")

with open(output_path, "w", encoding="utf-8") as f:
    f.write(clean_text)

print(f"[SUCCESS] Transcript saved to {output_path}")

[SUCCESS] Transcript saved to /home/kjh/workspace/ExtracTube/notebooks/outputs/tbndmgWiFJk.txt
